# Pancreatic cancer recurrence pipeline on Colab

This notebook clones the repository to Colab's local disk, installs its pinned environment, keeps the source data/results/checkpoints in Google Drive, stages the DICOM files temporarily on Colab's local SSD for speed, creates split-safe clinical feature tables, embeds CT reports with Qwen3-Embedding-8B, and creates a resumable SPECTRE image-embedding run.

Before running: choose **Runtime → Change runtime type → GPU**. The notebook creates `MyDrive/pc-recurrence-prediction/images` and `MyDrive/pc-recurrence-prediction/table`. Upload the one Excel workbook into `table/` for clinical and CT-report stages. To also run SPECTRE, upload one `.zip` archive containing the curated `dicom_selected` folder into `images/` (recommended). Patient data stays out of Git, but you should still confirm that using Colab complies with your data-handling requirements.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Edit these if you use a fork, tag/commit, or different Drive folder.
import csv
import subprocess
from datetime import UTC, datetime
from pathlib import Path

REPO_URL = "https://github.com/sezeriper/pc-recurrence-prediction.git"
REPO_REF = "master"
LOCAL_REPO = Path("/content/pc-recurrence-prediction")
DRIVE_PROJECT = Path("/content/drive/MyDrive/pc-recurrence-prediction")
IMAGES_DIR = DRIVE_PROJECT / "images"
TABLE_DIR = DRIVE_PROJECT / "table"
SPECTRE_RUN = DRIVE_PROJECT / "outputs/image_embeddings/spectre-colab"
# Clinical and text outputs get a fresh paired directory for every notebook run.
# Their manifests record the exact split and preprocessing/model provenance.
PIPELINE_RUN_LABEL = datetime.now(UTC).strftime("colab-%Y%m%dT%H%M%SZ")
CLINICAL_RUN = DRIVE_PROJECT / "outputs/clinical_features" / PIPELINE_RUN_LABEL
TEXT_RUN = DRIVE_PROJECT / "outputs/text_embeddings/qwen3-embedding-8b" / PIPELINE_RUN_LABEL
TEXT_MODEL_CACHE = DRIVE_PROJECT / ".cache/text_models"
SPLIT_SEED = 0
VALIDATION_FRACTION = 0.20
QWEN_BATCH_SIZE = 1
QWEN_MAX_LENGTH = 8192
for directory in (IMAGES_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print(f"Images folder: {IMAGES_DIR}")
print(f"Table folder:  {TABLE_DIR}")
print(f"Clinical run:  {CLINICAL_RUN}")
print(f"Text run:      {TEXT_RUN}")

In [ ]:
# Validate Drive inputs. A single .zip containing dicom_selected/ is preferred:
# it is much faster to copy one large archive from Drive than thousands of DICOM
# slices. The original unpacked layout remains supported as a fallback.
def data_status():
    workbooks = sorted(TABLE_DIR.rglob("*.xlsx"))
    archives = sorted(
        path for path in IMAGES_DIR.iterdir() if path.is_file() and path.suffix.lower() == ".zip"
    )
    image_files = [
        path
        for path in IMAGES_DIR.rglob("*")
        if path.is_file()
        and path.name != "curation_manifest.json"
        and path not in archives
    ]
    manifest = IMAGES_DIR / "curation_manifest.json"
    return workbooks, archives, image_files, manifest


workbooks, archives, image_files, manifest = data_status()
raw_dataset_ready = bool(image_files) and manifest.is_file()
archive_ready = len(archives) == 1
if len(workbooks) != 1:
    print("Data upload required:")
    print(f"  1. Upload exactly one .xlsx workbook to {TABLE_DIR} (found {len(workbooks)}).")
    input("After the upload finishes in Google Drive, press Enter to recheck. ")
    workbooks, archives, image_files, manifest = data_status()
    raw_dataset_ready = bool(image_files) and manifest.is_file()
    archive_ready = len(archives) == 1

if len(archives) > 1:
    raise FileExistsError(f"Expected at most one .zip archive in {IMAGES_DIR}, found: {archives}")
if len(workbooks) != 1:
    raise FileNotFoundError(f"Expected exactly one .xlsx workbook in {TABLE_DIR}.")
WORKBOOK = workbooks[0]
CURATED_DICOM = IMAGES_DIR
DICOM_ARCHIVE = archives[0] if archive_ready else None
DICOM_INPUT_AVAILABLE = archive_ready or raw_dataset_ready
print(f"Using workbook: {WORKBOOK}")
if DICOM_ARCHIVE is not None:
    print(f"Using DICOM archive: {DICOM_ARCHIVE}")
elif raw_dataset_ready:
    print(f"Found {len(image_files)} image file(s) in: {CURATED_DICOM}")
else:
    print(
        "No DICOM input found: clinical and text stages can run; "
        "SPECTRE will require DICOM input."
    )

## Clone and install
The URL must be readable without credentials. For a private repository, use your preferred GitHub authentication method and never paste a token into a shared notebook or commit it.

In [ ]:
if (LOCAL_REPO / ".git").is_dir():
    print(f"Updating existing checkout: {LOCAL_REPO}")
else:
    if LOCAL_REPO.exists():
        raise RuntimeError(f"{LOCAL_REPO} exists but is not a Git checkout.")
    !git clone "{REPO_URL}" "{LOCAL_REPO}"

# These are intentionally direct shell commands, so each setup step is visible.
!git -C "{LOCAL_REPO}" fetch --all --tags --prune
!git -C "{LOCAL_REPO}" checkout "{REPO_REF}"
if REPO_REF == "master":
    !git -C "{LOCAL_REPO}" pull --ff-only origin master
# Install uv through the active Colab interpreter. The project itself uses Python 3.12.
!python -m pip install --quiet uv
!uv python install 3.12
%cd {LOCAL_REPO}
!uv sync --python 3.12 --extra imaging --extra text

In [ ]:
# Use fast local disk for code and optional DICOM reads, but Drive for source
# data, checkpoints, and persistent outputs. Clinical/text stages need only the
# workbook; DICOM staging is deferred unless image input is available.
LOCAL_STAGE_ROOT = Path("/content/pc-recurrence-input-cache")
LOCAL_DICOM = LOCAL_STAGE_ROOT / "images"

# Everything under LOCAL_STAGE_ROOT is an expendable Colab-local cache. It is
# never a Drive path, so this cleanup cannot delete your Drive source data.
%cd {LOCAL_REPO}
RUN_DICOM = None
if DICOM_INPUT_AVAILABLE:
    !rm -rf "{LOCAL_STAGE_ROOT}"
    !mkdir -p "{LOCAL_STAGE_ROOT}"
    if DICOM_ARCHIVE is not None:
        LOCAL_ARCHIVE = LOCAL_STAGE_ROOT / "dicom_selected.zip"
        !rsync -a --info=progress2 "{DICOM_ARCHIVE}" "{LOCAL_ARCHIVE}"
        !unzip -q "{LOCAL_ARCHIVE}" -d "{LOCAL_STAGE_ROOT}/extracted"
        !rm -f "{LOCAL_ARCHIVE}"
        manifests = sorted((LOCAL_STAGE_ROOT / "extracted").rglob("curation_manifest.json"))
        if len(manifests) != 1:
            raise FileNotFoundError(
                "The archive must contain exactly one curation_manifest.json from dicom_selected/."
            )
        RUN_DICOM = manifests[0].parent
    else:
        !rsync -a --delete --info=progress2 "{CURATED_DICOM}/" "{LOCAL_DICOM}/"
        RUN_DICOM = LOCAL_DICOM
    if not (RUN_DICOM / "curation_manifest.json").is_file():
        raise FileNotFoundError(f"Local DICOM staging failed: {RUN_DICOM}")
    print(f"SPECTRE will read local DICOM files from: {RUN_DICOM}")
else:
    print("Skipping local DICOM staging because no image input is available.")

# Keep expensive, resumable model state and output artifacts on Drive.
def link_to_drive(name, target):
    local = LOCAL_REPO / name
    target.mkdir(parents=True, exist_ok=True)
    if local.is_symlink() and local.resolve() == target.resolve():
        return
    if local.is_symlink():
        local.unlink()
    elif local.exists():
        raise RuntimeError(f"{local} exists and is not a symlink.")
    local.symlink_to(target, target_is_directory=True)


for name in ("outputs", ".cache"):
    link_to_drive(name, DRIVE_PROJECT / name)
# Print the curated folder layout when image input is available. Missing or invalid
# patient folders are logged as skipped by pc-image-embed unless you add --require-all.
%cd {LOCAL_REPO}
if RUN_DICOM is not None:
    !find "{RUN_DICOM}" -mindepth 1 -maxdepth 1 -type d -printf "%f\n" | sort
!uv run python -c 'import torch; print("PyTorch:", torch.__version__)'
!nvidia-smi -L || echo "No GPU visible"

## Input data layout
`table/` must contain exactly one `.xlsx` workbook; that is sufficient for the clinical and CT-report stages. To run SPECTRE too, put one `.zip` archive containing the complete `dicom_selected/` folder in `images/` (recommended). The archive must include `curation_manifest.json` and the selected DICOM patient folders; an enclosing `dicom_selected/` directory is fine. The older unpacked layout is also supported: `images/` directly contains the selected patient folders and `curation_manifest.json`. This notebook does not run the DICOM selection/review workflow.

In [ ]:
# Example expected Drive layout:
# pc-recurrence-prediction/images/<patient-folder>/<selected-dicom-files>
# pc-recurrence-prediction/table/pankreas adeno ca 10 hasta.xlsx

## Create split-safe clinical feature tables
This stage reads the workbook once, makes a deterministic patient-level train/validation split, fits numerical normalization and categorical vocabularies on the training patients only, and writes separate raw and transformed tables. The raw tables contain CT report text, so keep this Drive project in an approved data-handling boundary. The following audit cell intentionally avoids displaying report text. Re-run the configuration cell to generate a fresh paired clinical/text output directory before intentionally creating a new split.

In [ ]:
clinical_command = [
    "uv",
    "run",
    "pc-clinical-data",
    "preprocess",
    "--workbook",
    str(WORKBOOK),
    "--run-dir",
    str(CLINICAL_RUN),
    "--validation-fraction",
    str(VALIDATION_FRACTION),
    "--split-seed",
    str(SPLIT_SEED),
]
print("Running:", " ".join(clinical_command))
subprocess.run(clinical_command, cwd=LOCAL_REPO, check=True)
if not (CLINICAL_RUN / "run_manifest.json").is_file():
    raise FileNotFoundError(f"Clinical preprocessing produced no manifest: {CLINICAL_RUN}")
print("Clinical feature artifacts:", CLINICAL_RUN)

## Embed CT reports with Qwen3-Embedding-8B (GPU)
This stage consumes the completed clinical run above and verifies its split-specific raw-table hashes before embedding. It writes distinct train and validation `.npz` vectors with 4,096-dimensional, L2-normalized embeddings. The first run downloads the pinned Qwen snapshot into the Drive-backed cache; it is a large 8B model, so use a high-memory GPU. Start with batch size 1 and the configured token limit; reduce `QWEN_MAX_LENGTH` if the selected Colab GPU runs out of memory. Reports with no text are skipped and recorded in the safe embedding audit rather than imputed.

In [ ]:
text_command = [
    "uv",
    "run",
    "--extra",
    "text",
    "pc-text-embed",
    "run",
    "--clinical-run",
    str(CLINICAL_RUN),
    "--run-dir",
    str(TEXT_RUN),
    "--model-cache",
    str(TEXT_MODEL_CACHE),
    "--batch-size",
    str(QWEN_BATCH_SIZE),
    "--max-length",
    str(QWEN_MAX_LENGTH),
    "--resume",
]
print("Running:", " ".join(text_command))
subprocess.run(text_command, cwd=LOCAL_REPO, check=True)
if not (TEXT_RUN / "run_manifest.json").is_file():
    raise FileNotFoundError(f"CT report embedding produced no manifest: {TEXT_RUN}")
print("CT report embedding artifacts:", TEXT_RUN)

## Audit the clinical split and report-embedding coverage
The audit prints split assignment and embedding status only. It does not display the `ct_report_text` column; inspect the raw tables only in an approved environment when necessary.

In [ ]:
def read_audit_csv(path, expected_columns):
    # Pipeline CSVs use UTF-8 with a BOM so they open cleanly in Excel.
    # utf-8-sig removes that BOM before DictReader sees the first header.
    with path.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        actual_columns = set(reader.fieldnames or ())
        missing = expected_columns - actual_columns
        if missing:
            raise ValueError(f"{path.name} is missing expected columns: {sorted(missing)}")
        return list(reader)

assignments = read_audit_csv(
    CLINICAL_RUN / "split_assignments.csv",
    {"patient_id", "split", "target_recurrence"},
)
for split in ("train", "validation"):
    patients = [row["patient_id"] for row in assignments if row["split"] == split]
    labels = [row["target_recurrence"] for row in assignments if row["split"] == split]
    print(
        f"{split}: {len(patients)} patient(s), recurrence labels={labels}, "
        f"patient IDs={patients}"
    )

summary_path = TEXT_RUN / "embedding_summary.csv"
if summary_path.is_file():
    summaries = read_audit_csv(summary_path, {"split", "status"})
    for split in ("train", "validation"):
        rows = [row for row in summaries if row["split"] == split]
        embedded = sum(row["status"] == "complete" for row in rows)
        skipped = sum(row["status"] == "skipped" for row in rows)
        print(f"{split}: {embedded} embedded, {skipped} skipped; no report text displayed.")
else:
    print(f"No CT report embedding audit exists at: {summary_path}")
    print("Run the Qwen embedding cell above; it will now stop at the original error if it fails.")

feature_path = CLINICAL_RUN / "clinical_features_train.csv"
with feature_path.open(newline="", encoding="utf-8-sig") as handle:
    clinical_columns = next(csv.reader(handle))
print("Clinical feature columns:", clinical_columns)
if summary_path.is_file():
    print("Training text vectors:", TEXT_RUN / "text_embeddings_train.npz")
    print("Validation text vectors:", TEXT_RUN / "text_embeddings_validation.npz")
else:
    print("Text-vector files are unavailable until the Qwen stage completes.")

## Generate SPECTRE embeddings (GPU)
The stable Drive-backed run directory plus `--resume` allows a compatible interrupted run to continue. The first run downloads pinned weights to the persistent cache. `pc-image-embed` prints its checkpoint, device, patient-by-patient, cache, skip, and completion progress below. SPECTRE weights are CC-BY-NC-SA and restricted to non-commercial use.

In [ ]:
%cd {LOCAL_REPO}
if RUN_DICOM is None:
    raise FileNotFoundError(
        "SPECTRE needs DICOM input. Upload the selected archive or curated folder "
        "to images/, then rerun setup."
    )
# Agg prevents Colab's notebook-only Matplotlib backend from breaking SPECTRE imports.
!MPLBACKEND=Agg uv run pc-image-embed run --encoder spectre \
    --dicom-root "{RUN_DICOM}" \
    --workbook "{WORKBOOK}" \
    --run-dir "{SPECTRE_RUN}" --resume
print("SPECTRE embedding artifacts:", SPECTRE_RUN)

## What this produces
The clinical run contains separate raw and processed train/validation CSV tables, split assignments, fitted preprocessing parameters, and a manifest. The paired Qwen run contains `text_embeddings_train.npz`, `text_embeddings_validation.npz`, a report-safe `embedding_summary.csv`, and a model/provenance manifest. The SPECTRE run contains `image_embeddings.npz`, `patch_embeddings.npz`, `embedding_summary.csv`, and `run_manifest.json`. These stages are intentionally separate and auditable: the current `pc-recurrence-classify train` CLI still requires both Merlin and SPECTRE image-embedding runs and does not yet fuse clinical or text vectors.

## Optional repository verification
Uncomment these commands if you modify code in Colab.

In [ ]:
# %cd {LOCAL_REPO}
# !uv sync --python 3.12 --extra imaging --extra text --group dev
# !uv run ruff check .
# !uv run pytest